# 00 - Gaussian Process Regression Temelleri

Bu notebook'un amacı `GaussianProcessRegressor` sınıfını Bayes optimizasyonundan önce bağımsız olarak anlamaktır.

Temel ayrım:

- **Gaussian Process Regression (GPR):** bir regresyon ve belirsizlik modelidir.
- **Bayes optimizasyonu:** GPR gibi bir surrogate modelin tahmin ortalaması ve belirsizliğini kullanarak yeni deney noktaları seçen optimizasyon yaklaşımıdır.

Bu nedenle `sklearn.gaussian_process.GaussianProcessRegressor` tek başına bir black-box optimizer değildir.

## 1. Gaussian Process ne üretir?

Bir giriş noktası \(x\) için GP yalnızca tek bir tahmin vermez. Tipik olarak:

\[
\mu(x)
\]

tahmin ortalamasını ve

\[
\sigma(x)
\]

tahmin belirsizliğini üretir.

Bayes optimizasyonunda kritik olan nokta budur. Çünkü optimizer yalnızca "nerede düşük değer bekliyorum?" sorusuna değil, aynı zamanda "nerede yeterince belirsizlik var?" sorusuna da bakabilir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern

## 2. Örnek fonksiyon

Aşağıdaki fonksiyon yalnızca öğretim amacıyla seçilmiştir. Gerçek bir üretim prosesi değildir.

Az sayıda noktayı gözleyip GP'nin aradaki fonksiyonu ve belirsizliği nasıl modellediğini inceleyeceğiz.

In [ ]:
def gercek_fonksiyon(x):
    return np.sin(1.5 * x) + 0.15 * x

X_egitim = np.array([[-2.5], [-1.0], [0.2], [1.8], [3.0]])
y_egitim = gercek_fonksiyon(X_egitim[:, 0])

X_grid = np.linspace(-3.0, 3.5, 400).reshape(-1, 1)

## 3. Kernel seçimi

Gaussian Process'te kernel, iki giriş noktasının ne kadar benzer davranmasını beklediğimizi tanımlar.

Burada Matérn kernel kullanıyoruz:

```python
Matern(nu=2.5)
```

Scikit-learn dokümantasyonundaki standart yorumda:

- `nu=1.5` yaklaşık bir kez türevlenebilir fonksiyonlar,
- `nu=2.5` yaklaşık iki kez türevlenebilir fonksiyonlar

için yaygın ara değerlerdir.

Daha küçük `nu`, daha pürüzlü fonksiyonlara izin verir. Çok büyük `nu` değerlerinde Matérn, RBF kernel davranışına yaklaşır.

In [ ]:
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(
    length_scale=1.0,
    length_scale_bounds=(1e-3, 1e3),
    nu=2.5,
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-8,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=42,
)

gp.fit(X_egitim, y_egitim)

ortalama, std = gp.predict(X_grid, return_std=True)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(X_grid[:, 0], gercek_fonksiyon(X_grid[:, 0]), label="Gerçek fonksiyon")
plt.plot(X_grid[:, 0], ortalama, label="GP tahmini")
plt.fill_between(
    X_grid[:, 0],
    ortalama - 1.96 * std,
    ortalama + 1.96 * std,
    alpha=0.2,
    label="Yaklaşık %95 belirsizlik bandı",
)
plt.scatter(X_egitim[:, 0], y_egitim, label="Gözlemler")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Gaussian Process Regression")
plt.legend()
plt.grid(True)
plt.show()

## 4. Grafiği nasıl okumalıyız?

Gözlem noktalarına yakın bölgelerde belirsizlik genellikle azalır. Gözlemden uzak bölgelerde ise standart sapma büyür.

Bayes optimizasyonunda acquisition function bu iki bilgiyi birleştirir:

- düşük tahmini amaç değeri: **exploitation**
- yüksek belirsizlik: **exploration**

Bu exploration-exploitation dengesi, pahalı deneylerin rastgele yapılması yerine bilgi değeri yüksek noktalara yönelmesini sağlar.

## 5. `alpha` ve `WhiteKernel` konusu

İki kavram birbirine karıştırılmamalıdır.

`alpha`:

- kernel matrisinin köşegenine eklenir,
- sayısal kararlılık sağlayabilir,
- bilinen gözlem gürültüsü varyansını temsil etmek için de kullanılabilir.

`WhiteKernel`:

- kernelin açık bir gürültü bileşenidir,
- gürültü seviyesinin model tarafından öğrenilmesine izin verebilir.

Her ikisini birlikte kullanmak teknik olarak mümkündür; fakat neyi temsil ettiklerini açıkça belirlemek gerekir. Eğitim örneklerinde gereksiz çift gürültü modellemesinden kaçınmak daha temizdir.

## 6. GPR'nin sınırlamaları

Klasik Gaussian Process modelleri eğitim örneği sayısı büyüdükçe pahalı hale gelir. Standart yaklaşımın temel lineer cebiri yaklaşık \(O(n^3)\) maliyetlidir.

Bu nedenle GP özellikle:

- veri sayısının sınırlı,
- her yeni gözlemin pahalı,
- belirsizliğin önemli

olduğu problemlerde güçlüdür.

Binlerce veya milyonlarca ucuz veri noktasında klasik GP çoğu zaman ilk tercih değildir.

## 7. Sonraki notebook

Bir sonraki notebook'ta bu GP modelini surrogate model olarak kullanıp Expected Improvement ve Lower Confidence Bound ile gerçek bir Bayes optimizasyonu döngüsü kuracağız.